In [ ]:
import numpy as np
import pandas as pd

from ranx import Qrels, Run, evaluate
from common import embeddings as E
from common.corpus import load_corpus
from common.goldset import load_queries, qrels
from common import retrieval as R


docs = load_corpus()
chunks = R.chunk_corpus(docs)
queries = load_queries()
gold = Qrels(qrels())

bm25 = R.BM25Retriever(chunks)
dense = R.DenseRetriever(chunks)



In [6]:
# Reciprocal Rank Fusion (RRF)

def rrf(rankings: list[list[str]], k: int = 60): # liste aus chunk_id
    fused = {}
    for ranking in rankings:
        for rang, cid in enumerate(ranking, start=1):
            fused[cid] = fused.get(cid,0.0) + 1.0/(k+rang) # Summation über dict keys
    return fused 
        
by_id = {c.chunk_id: c for c in chunks}


def hybrid_search(query, k=10):
    bm = [c.chunk_id for c, _ in bm25.search(query, k)]
    dn = [c.chunk_id for c, _ in dense.search(query, k)]

    fused = rrf([bm, dn])
    top = sorted(fused.items(), key = lambda x: -x[1])[:k]
    return [(by_id[cid], s) for cid,s in top]

def hybrid_rerank(query, k = 10, pool = 20): # in Praxis pool = 100
    candidates = hybrid_search(query, k = pool) # verschwenderisch Holen
    return R.cross_encoder_rerank(query, candidates, k = k) # verdichten 


In [7]:
import time

def run_for(searcher) -> dict:
    out = {}
    for q in queries:
        scored = searcher(q.text)
        out[q.qid] = {d: s for d, s in R.collapse_to_docs(scored)}
    return out


setups = {
    "BM25":         lambda q: bm25.search(q, 10),
    "Dense":        lambda q: dense.search(q, 10),
    "Hybrid-RRF":   lambda q: hybrid_search(q, 10),
    "Hybrid+Rerank": lambda q: hybrid_rerank(q, 10),
}

zeilen = []
laeufe = {}
for name, fn in setups.items():
    t0 = time.perf_counter()
    run = run_for(fn)
    dt = time.perf_counter() - t0
    laeufe[name] = run
    m = evaluate(gold, Run(run, name=name), ["ndcg@10", "recall@10", "mrr"])
    zeilen.append({"setup": name, **{k: round(float(v), 3) for k, v in m.items()},
                   "sek/Query": round(dt / len(queries), 3)})

df = pd.DataFrame(zeilen).set_index("setup")
print(df)


               ndcg@10  recall@10    mrr  sek/Query
setup                                              
BM25             0.935      0.958  0.958      0.000
Dense            0.874      0.958  0.892      0.035
Hybrid-RRF       0.963      0.958  1.000      0.035
Hybrid+Rerank    0.946      1.000  0.958      1.075


In [10]:
# Wo gewinnt was?
# Wir suchen Queries bei denen BM25 und Dense unterschiedlich gut sind?

def ndcg_pro_query(run, qid):
    return float(evaluate(Qrels({qid: qrels()[qid]}), Run({qid: run[qid]}), ["ndcg@10"]))


print(f"{'qid':4s} {'BM25':>6s} {'Dense':>6s}  Frage")
for q in queries:
    nb = ndcg_pro_query(laeufe["BM25"], q.qid)
    nd = ndcg_pro_query(laeufe["Dense"], q.qid)
    if abs(nb - nd) > 0.1:
        print(f"{q.qid:4s} {nb:6.2f} {nd:6.2f}  {q.text[:46]}")

qid    BM25  Dense  Frage
q07    0.52   0.96  Wie wird die Pumpe in Betrieb genommen und ent
q08    0.90   0.66  Wie oft muss der Rücklauffilter getauscht werd
q09    1.00   0.63  Welche Teile-Nummer hat der Wellendichtring?
q11    0.92   0.32  Was tun bei Kavitation (pfeifendes Geräusch)?


In [9]:
def hybrid_k(query, kk, pool=20):
    bm = [c.chunk_id for c, _ in bm25.search(query, pool)]
    dn = [c.chunk_id for c, _ in dense.search(query, pool)]
    top = sorted(rrf([bm, dn], k=kk).items(), key=lambda x: -x[1])[:10]
    return [(by_id[cid], s) for cid, s in top]


for kk in (10, 60, 200):
    run = run_for(lambda q, kk=kk: hybrid_k(q, kk))
    ndcg = float(evaluate(gold, Run(run), ["ndcg@10"]))
    print(f"rrf k={kk:3d}: nDCG@10={ndcg:.3f}")


rrf k= 10: nDCG@10=0.964
rrf k= 60: nDCG@10=0.963
rrf k=200: nDCG@10=0.963
